In [1]:
import yaml
from utils import set_seed, load_graph_data

with open(r'C:/Users/Damir/Desktop/GNN_dataset_test/GNN_dataset_test/params.yaml') as f:
    cfg = yaml.safe_load(f)

configs_model = cfg["model"]
configs_train = cfg["train"]
configs_paths = cfg["paths"]
configs_preproc = cfg["preprocess"]

set_seed(configs_train["seed"])

train_dataloader, val_dataloader, data_features = load_graph_data(configs_paths, configs_preproc, configs_train, return_val = True)

hetero_graph_batch, sample_names = next(iter(train_dataloader))
hetero_graph_batch, sample_names

c:\Users\Damir\Desktop\GNN_dataset_test\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Метаданные (train часть):
       MODEL    STATUS                                                MIN  \
0  e1_v00004  COMPLETE  [3.6298862e+00 2.9635735e-02 3.0059695e-01 0.0...   
1  e1_v00001  COMPLETE  [5.1691298e+00 2.9635735e-02 3.0256003e-01 0.0...   
2  e1_v00008  COMPLETE  [2.5237122e+00 2.9635735e-02 3.1039816e-01 0.0...   
3  e1_v00007  COMPLETE  [4.4991727e+00 2.9635735e-02 3.0111608e-01 0.0...   
4  e1_v00005  COMPLETE  [3.1294053e+00 2.9635735e-02 3.0619934e-01 0.0...   
5  e1_v00002  COMPLETE  [5.0439239e+00 2.9635735e-02 3.0502006e-01 0.0...   
6  e1_v00010  COMPLETE  [5.8906302e+00 2.9635735e-02 3.0328861e-01 0.0...   
7  e1_v00009  COMPLETE  [3.0328267e+00 2.9635735e-02 3.0391216e-01 0.0...   
8  e1_v00006  COMPLETE  [1.5860901e+00 2.9635735e-02 3.0538654e-01 0.0...   
9  e1_v00003  COMPLETE  [2.2270913e+00 2.9635735e-02 3.0510870e-01 0.0...   

                                                 MAX  \
0  [2.3525731e+02 2.9750499e-01 1.0000000e+00 6.9...   
1  [3.5139566e

(HeteroDataBatch(
   cell={
     x=[19440, 8],
     labels=[19440],
     batch=[19440],
     ptr=[4],
   },
   well={
     x=[32, 1],
     y=[32, 3, 25],
     batch=[32],
     ptr=[4],
   },
   (cell, flows_to, cell)={
     edge_index=[2, 38133],
     edge_attr=[3],
   },
   (cell, linked_to, well)={ edge_index=[2, 495] }
 ),
 ('e1_v00008.pt', 'e1_v00003.pt', 'e1_v00010.pt'))

In [2]:
import torch
import yaml
import os
import matplotlib.pyplot as plt
from torch_geometric.loader import DataLoader

# Импортируем твою модель и функции
from simple_model import SimpleHeteroGNN
from utils import load_graph_data

# 1. Загрузка параметров из yaml
with open("params.yaml", "r") as f:
    config = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Работаем на: {device}")

# 2. Загрузка данных через твою функцию
# Она сама сделает split, посчитает статистики и нормализует всё
train_loader, val_loader, feat_list = load_graph_data(
    config['paths'], 
    config['preprocess'], 
    config['train']
)

# 3. Инициализация модели
# Берем один пример из базы, чтобы узнать размерности входных данных
sample_data, _ = next(iter(train_loader))

model = SimpleHeteroGNN(
    cell_features=sample_data['cell'].x.size(1),
    well_features=sample_data['well'].x.size(1),
    hidden_dim=config['model']['nz'], # используем nz из yaml
    out_seq_len=25,
    num_phases=3
).to(device)

Работаем на: cpu
Метаданные (train часть):
       MODEL    STATUS                                                MIN  \
0  e1_v00004  COMPLETE  [3.6298862e+00 2.9635735e-02 3.0059695e-01 0.0...   
1  e1_v00001  COMPLETE  [5.1691298e+00 2.9635735e-02 3.0256003e-01 0.0...   
2  e1_v00008  COMPLETE  [2.5237122e+00 2.9635735e-02 3.1039816e-01 0.0...   
3  e1_v00007  COMPLETE  [4.4991727e+00 2.9635735e-02 3.0111608e-01 0.0...   
4  e1_v00005  COMPLETE  [3.1294053e+00 2.9635735e-02 3.0619934e-01 0.0...   
5  e1_v00002  COMPLETE  [5.0439239e+00 2.9635735e-02 3.0502006e-01 0.0...   
6  e1_v00010  COMPLETE  [5.8906302e+00 2.9635735e-02 3.0328861e-01 0.0...   
7  e1_v00009  COMPLETE  [3.0328267e+00 2.9635735e-02 3.0391216e-01 0.0...   
8  e1_v00006  COMPLETE  [1.5860901e+00 2.9635735e-02 3.0538654e-01 0.0...   
9  e1_v00003  COMPLETE  [2.2270913e+00 2.9635735e-02 3.0510870e-01 0.0...   

                                                 MAX  \
0  [2.3525731e+02 2.9750499e-01 1.0000000e+00 6.9... 

c:\Users\Damir\Desktop\GNN_dataset_test\GNN_dataset_test\simple_model.py:22: UserWarning: There exist node types ({'cell'}) whose representations do not get updated during message passing as they do not occur as destination type in any edge type. This may lead to unexpected behavior.
  self.well_conv = HeteroConv({


In [3]:
if hasattr(sample_data['cell', 'flows_to', 'cell'], 'edge_attr'):
    print("edge_attr присутствует для cell->cell")
    edge_attr = sample_data['cell', 'flows_to', 'cell'].edge_attr
    print("Форма edge_attr:", edge_attr.shape)
else:
    print("edge_attr ОТСУТСТВУЕТ для cell->cell")

edge_attr присутствует для cell->cell


AttributeError: 'list' object has no attribute 'shape'

In [4]:
print("Атрибуты cell->cell:", sample_data['cell', 'flows_to', 'cell'].keys())

Атрибуты cell->cell: KeysView({'edge_index': tensor([[    0,     0,     1,  ..., 19436, 19437, 19438],
        [    1,    24,     2,  ..., 19437, 19438, 19439]]), 'edge_attr': [array([0.1292584 , 0.13061469, 0.        , ..., 0.        , 0.        ,
       0.        ], shape=(12648,)), array([0.75881842, 0.70728313, 0.74884485, ..., 0.6637013 , 0.70735733,
       0.75847108], shape=(12837,)), array([0.26960303, 0.2538296 , 0.        , ..., 0.        , 0.        ,
       0.        ], shape=(12648,))]})


In [5]:
sample_data

HeteroDataBatch(
  cell={
    x=[19440, 8],
    labels=[19440],
    batch=[19440],
    ptr=[4],
  },
  well={
    x=[32, 1],
    y=[32, 3, 25],
    batch=[32],
    ptr=[4],
  },
  (cell, flows_to, cell)={
    edge_index=[2, 38133],
    edge_attr=[3],
  },
  (cell, linked_to, well)={ edge_index=[2, 495] }
)

In [10]:
edge_attr_cell = sample_data['cell', 'flows_to', 'cell'].get('edge_attr')
if edge_attr_cell is not None:
    print("edge_attr найден!")
else:
    print("edge_attr отсутствует")
edge_attr_well = sample_data['cell', 'linked_to', 'well'].get('edge_attr')
if edge_attr_well is not None:
    print("edge_attr найден!")
else:
    print("edge_attr отсутствует")

edge_attr найден!
edge_attr отсутствует


In [7]:
sample_data['cell'].x[sample_data['cell'].batch == 0].shape

torch.Size([6480, 8])

In [8]:
model(sample_data).shape

torch.Size([32, 3, 25])

In [9]:
sample_data['cell', 'flows_to', 'cell'].edge_index
sample_data['cell', 'linked_to', 'well'].edge_index

tensor([[  105,   969,  1401,  1833,  2697,  3129,  3993,  4425,  5289,  5721,
           225,   657,  1089,  1953,  2385,  3249,  4113,  4545,  4977,  5409,
           110,   542,   974,  1406,  1838,  3134,  3566,  3998,  4430,  4862,
          5294,  5726,  6158,   350,   782,  1214,  1646,  2942,  3374,  3806,
          4238,  5102,  5534,  6398,   230,  1094,  1526,  1958,  2390,  2822,
          3254,  4118,  5414,  6665,  7097,  7529,  7961,  8393,  8825,  9257,
          9689, 10121, 10553, 10985, 11417, 11849, 12281, 12713,  6590,  7022,
          7454,  7886,  8318,  8750,  9614, 10046, 10478, 10910, 11342, 11774,
         12206, 12638,  6737,  7169,  7601,  8033,  8465,  8897,  9329,  9761,
         10193, 10625, 11057, 11489, 12353, 12785,  6734,  7166,  7598,  8030,
          8462,  8894,  9326,  9758, 10622, 11054, 11486, 11918, 12782,  6830,
          7262,  7694,  8126,  8990,  9422,  9854, 10286, 10718, 11582, 12014,
         12878,  6826,  7690,  8122,  8554,  8986,  

In [12]:
sample_data['cell', 'linked_to', 'well'].edge_index

tensor([[  105,   969,  1401,  1833,  2697,  3129,  3993,  4425,  5289,  5721,
           225,   657,  1089,  1953,  2385,  3249,  4113,  4545,  4977,  5409,
           110,   542,   974,  1406,  1838,  3134,  3566,  3998,  4430,  4862,
          5294,  5726,  6158,   350,   782,  1214,  1646,  2942,  3374,  3806,
          4238,  5102,  5534,  6398,   230,  1094,  1526,  1958,  2390,  2822,
          3254,  4118,  5414,  4679,  5111,  5543,  5975,  6407,  6839,  7271,
          7703,  8135,  8567,  8999,  9431,  9863, 10295, 10727,  4604,  5036,
          5468,  5900,  6332,  6764,  7628,  8060,  8492,  8924,  9356,  9788,
         10220, 10652,  4751,  5183,  5615,  6047,  6479,  6911,  7343,  7775,
          8207,  8639,  9071,  9503, 10367, 10799,  4748,  5180,  5612,  6044,
          6476,  6908,  7340,  7772,  8636,  9068,  9500,  9932, 10796,  4844,
          5276,  5708,  6140,  7004,  7436,  7868,  8300,  8732,  9596, 10028,
         10892,  4840,  5704,  6136,  6568,  7000,  

In [ ]:
x_dict = {
    'cell': model.cell_emb(data['cell'].x),
    'well': model.well_emb(data['well'].x)
}

# Связи
c2c_edge_index = data['cell', 'flows_to', 'cell'].edge_index
edge_index_dict = data.edge_index_dict

# Шаг 1: Пропаганда внутри пласта (ячейка -> ячейка)
h_cell = x_dict['cell']
for conv in model.cell_convs:
    h_cell = conv(h_cell, c2c_edge_index)
    h_cell = F.relu(h_cell)

x_dict['cell'] = h_cell

# Шаг 2: Сбор информации на скважины (ячейка -> скважина)
# well_updates получит эмбеддинги для узлов-приемников (скважин)
well_updates = model.well_conv(x_dict, edge_index_dict)
h_well = well_updates['well']

# Шаг 3: Финальный слой
out = model.well_mlp(h_well)
out.view(-1, 3, 25)

NameError: name 'self' is not defined